Design and implement a comprehensive model versioning system, integrating tools like MLflow or DVC to track, manage, and roll back model versions while maintaining reproducibility and auditability.

In [1]:
import os
import shutil
import hashlib
import json
import time
import numpy as np
import pandas as pd
import joblib

In [2]:
from datetime import datetime
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

In [40]:
class ClinicalModelRegistry:
    """
    Lightweight, audit-compliant model versioning and registry system.
    Handles run logging, artifact hashing, stage transitions, and rollbacks.
    """
    def __init__(self, registry_dir="model_registry"):
        self.registry_dir = registry_dir
        self.manifest_path = os.path.join(self.registry_dir, "registry_manifest.json")
        os.makedirs(self.registry_dir, exist_ok=True)
        self._init_manifest()

    def _init_manifest(self):
        if not os.path.exists(self.manifest_path):
            manifest = {
                "active_production_version": None,
                "versions": {}
            }
            self._save_manifest(manifest)

    def _load_manifest(self):
        with open(self.manifest_path, "r") as f:
            return json.load(f)

    def _save_manifest(self, manifest):
        with open(self.manifest_path, "w") as f:
            json.dump(manifest, f, indent=4)

    def _compute_hash(self, file_path):
        hasher = hashlib.sha256()
        with open(file_path, "rb") as f:
            while chunk := f.read(8192):
                hasher.update(chunk)
        return hasher.hexdigest()

    def log_and_register_run(self, model_pipeline, metrics: dict, params: dict, feature_cols: list) -> str:
        """Logs model artifacts, parameters, metrics, and generates a new semantic version tag."""
        manifest = self._load_manifest()
        version_num = len(manifest["versions"]) + 1
        version_tag = f"v{version_num}.0.0"
        
        version_dir = os.path.join(self.registry_dir, version_tag)
        os.makedirs(version_dir, exist_ok=True)

        artifact_path = os.path.join(version_dir, "model_pipeline.joblib")
        joblib.dump(model_pipeline, artifact_path, compress=3)
        
        sha256_hash = self._compute_hash(artifact_path)

        metadata = {
            "version": version_tag,
            "timestamp": datetime.now().isoformat(),
            "stage": "Staging",
            "sha256_checksum": sha256_hash,
            "features": feature_cols,
            "parameters": params,
            "metrics": metrics
        }

        meta_path = os.path.join(version_dir, "metadata.json")
        with open(meta_path, "w") as f:
            json.dump(metadata, f, indent=4)

        manifest["versions"][version_tag] = metadata
        self._save_manifest(manifest)
        
        print(f"[REGISTER] Successfully logged run -> Version: {version_tag} | Stage: Staging | Checksum: {sha256_hash[:8]}...")
        return version_tag

    def transition_stage(self, version_tag: str, new_stage: str):
        """Transitions a model version between Staging, Production, and Archived."""
        valid_stages = ["Staging", "Production", "Archived"]
        if new_stage not in valid_stages:
            raise ValueError(f"Invalid stage. Select from {valid_stages}")

        manifest = self._load_manifest()
        if version_tag not in manifest["versions"]:
            raise KeyError(f"Version '{version_tag}' not found in registry.")

        if new_stage == "Production":
            # Demote current Production model to Archived
            current_prod = manifest.get("active_production_version")
            if current_prod and current_prod in manifest["versions"]:
                manifest["versions"][current_prod]["stage"] = "Archived"
                print(f"[TRANSITION] Demoted current Production model '{current_prod}' -> Archived")
            
            manifest["active_production_version"] = version_tag

        manifest["versions"][version_tag]["stage"] = new_stage
        self._save_manifest(manifest)
        print(f"[TRANSITION] Model '{version_tag}' promoted to -> {new_stage}")
        

        
        

In [ ]:
def transition_stage(self, version_tag: str, new_stage: str):
        """Transitions a model version between Staging, Production, and Archived."""
        valid_stages = ["Staging", "Production", "Archived"]
        if new_stage not in valid_stages:
            raise ValueError(f"Invalid stage. Select from {valid_stages}")

        manifest = self._load_manifest()
        if version_tag not in manifest["versions"]:
            raise KeyError(f"Version '{version_tag}' not found in registry.")

        if new_stage == "Production":
            # Demote current Production model to Archived
            current_prod = manifest.get("active_production_version")
            if current_prod and current_prod in manifest["versions"]:
                manifest["versions"][current_prod]["stage"] = "Archived"
                print(f"[TRANSITION] Demoted current Production model '{current_prod}' -> Archived")
            
            manifest["active_production_version"] = version_tag

        manifest["versions"][version_tag]["stage"] = new_stage
        self._save_manifest(manifest)
        print(f"[TRANSITION] Model '{version_tag}' promoted to -> {new_stage}")

In [ ]:
def rollback_production(self, target_version: str):
        """Rolls back the active Production pointer to a target model version."""
        print(f"[ROLLBACK] Initiating rollback to model version: {target_version}...")
        self.transition_stage(target_version, "Production")

In [ ]:
def load_production_model(self):
        """Loads and verifies the active Production model artifact."""
        manifest = self._load_manifest()
        prod_version = manifest.get("active_production_version")
        
        if not prod_version:
            raise RuntimeError("No model is currently promoted to Production.")

        prod_meta = manifest["versions"][prod_version]
        artifact_path = os.path.join(self.registry_dir, prod_version, "model_pipeline.joblib")

        # Verify integrity check before loading
        current_hash = self._compute_hash(artifact_path)
        if current_hash != prod_meta["sha256_checksum"]:
            raise SecurityError(f"CRITICAL: Integrity failure for {prod_version}! Checksum mismatch detected.")

        model = joblib.load(artifact_path)
        print(f"[LOAD] Successfully loaded active Production model '{prod_version}' (Integrity Verified)")
        return model, prod_meta

In [ ]:
def get_registry_summary(self) -> pd.DataFrame:
        """Returns a structured audit table of all model versions in the registry."""
        manifest = self._load_manifest()
        records = []
        for v, meta in manifest["versions"].items():
            records.append({
                "Version": v,
                "Stage": meta["stage"],
                "Timestamp": meta["timestamp"][:19],
                "ROC-AUC": meta["metrics"].get("roc_auc"),
                "F1-Score": meta["metrics"].get("f1_score"),
                "Learning Rate": meta["parameters"].get("learning_rate"),
                "Max Depth": meta["parameters"].get("max_depth"),
                "SHA-256 (Truncated)": meta["sha256_checksum"][:10] + "..."
            })
        return pd.DataFrame(records)

In [22]:
registry = ClinicalModelRegistry(registry_dir="clinical_model_registry")

In [23]:
np.random.seed(42)
n_features = 8
X_raw, y_raw = make_classification(
    n_samples=1200, n_features=n_features, n_informative=5, random_state=42
)

In [24]:
feature_cols = [f"biomarker_{i+1}" for i in range(n_features)]
X = pd.DataFrame(X_raw, columns=feature_cols)
y = y_raw

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
preprocessor = ColumnTransformer([('scaler', StandardScaler(), feature_cols)])

In [26]:
params_v1 = {"learning_rate": 0.01, "max_depth": 3, "max_iter": 50}
pipe_v1 = Pipeline([('preprocessor', preprocessor), ('classifier', HistGradientBoostingClassifier(**params_v1, random_state=42))])
pipe_v1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                                  ['biomarker_1', 'biomarker_2',
                                                   'biomarker_3', 'biomarker_4',
                                                   'biomarker_5', 'biomarker_6',
                                                   'biomarker_7',
                                                   'biomarker_8'])])),
                ('classifier',
                 HistGradientBoostingClassifier(learning_rate=0.01, max_depth=3,
                                                max_iter=50,
                                                random_state=42))])

In [27]:
probs_v1 = pipe_v1.predict_proba(X_test)[:, 1]
preds_v1 = pipe_v1.predict(X_test)
metrics_v1 = {"roc_auc": round(roc_auc_score(y_test, probs_v1), 4), "f1_score": round(f1_score(y_test, preds_v1), 4)}


In [29]:
tag_v1 = registry.log_and_register_run(pipe_v1, metrics_v1, params_v1, feature_cols)

registry.transition_stage(tag_v1, "Production")

[REGISTER] Successfully logged run -> Version: v5.0.0 | Stage: Staging | Checksum: 6b29e081...
[TRANSITION] Demoted current Production model 'v4.0.0' -> Archived
[TRANSITION] Model 'v5.0.0' promoted to -> Production


In [30]:
params_v2 = {"learning_rate": 0.08, "max_depth": 6, "max_iter": 120}
pipe_v2 = Pipeline([('preprocessor', preprocessor), ('classifier', HistGradientBoostingClassifier(**params_v2, random_state=42))])
pipe_v2.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                                  ['biomarker_1', 'biomarker_2',
                                                   'biomarker_3', 'biomarker_4',
                                                   'biomarker_5', 'biomarker_6',
                                                   'biomarker_7',
                                                   'biomarker_8'])])),
                ('classifier',
                 HistGradientBoostingClassifier(learning_rate=0.08, max_depth=6,
                                                max_iter=120,
                                                random_state=42))])

In [31]:
probs_v2 = pipe_v2.predict_proba(X_test)[:, 1]
preds_v2 = pipe_v2.predict(X_test)
metrics_v2 = {"roc_auc": round(roc_auc_score(y_test, probs_v2), 4), "f1_score": round(f1_score(y_test, preds_v2), 4)}

In [32]:
tag_v2 = registry.log_and_register_run(pipe_v2, metrics_v2, params_v2, feature_cols)
registry.transition_stage(tag_v2, "Production")

[REGISTER] Successfully logged run -> Version: v6.0.0 | Stage: Staging | Checksum: 834b41ec...
[TRANSITION] Demoted current Production model 'v5.0.0' -> Archived
[TRANSITION] Model 'v6.0.0' promoted to -> Production
